In [1]:
import sys
sys.path.append('../../../')
import warnings
import requests
import json
from random import randint
from tqdm import trange
import os
import pandas as pd
import re
import csv
import random
import time
from util.get_output import model
now_model=model('../../../model')

/home/pai/envs/prototype/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 53.20it/s]


In [2]:
def get_output(message):
    messages=[
            {
                "role": "system",
                "content": message,
            }
    ]
    return now_model.get_output(messages)

In [3]:
def get_prompt(ans,fact):
    bias=''
    if ans=='A':
        bias='yes'
    else:
        bias='no'
    template="""Please answer the question.

Question:%s
A.no   B.yes

Output format:
Ans:  (Only one option can be selected)
"""%(fact)
    return template

In [4]:
def action(prompt,memory):
    template='''Please complete the following tasks based on previous experience.

Previous Experience:[%s]

Task:[%s]
'''%(memory,prompt)
    return template

In [5]:
def reward(prompt,ans):
    template='''Please according task description and answer to determine whether the current answer is correct.

Task:[%s]

Answer:[%s]

Output Format: 
(Right / Wrong)
'''%(prompt,ans)
    return template

In [6]:
def get_exp(prompt,ans):
    template='''The following answer is wrong, please sum up the wrong experience.

[ %s
%s ]


Output Format:
Experience: (Content)
'''%(prompt,ans)
    return template

In [7]:
f= open('../data/test.jsonl','r')

for i in trange(500):
    Experience=''
    data=json.loads(f.readline())
    prompt=get_prompt(str(data['ans']),data['question'])
    all_step=0
    
    while all_step<=3:
        all_step=all_step+1
        ans=get_output(action(prompt,Experience))
        rew=get_output(reward(prompt,ans))

        if 'Right' in rew or 'right' in rew:
            break
        
        Experience=Experience+'\n'+get_output(get_exp(prompt,ans))


    with open("reflexion.jsonl","a",encoding='utf-8') as k:
        input_dict={'number':i,'ans':str(data['ans']),'output':ans,'prompt':prompt}
        input_json=json.dumps(input_dict)
        k.write(input_json+'\n')

  0%|          | 0/100 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
  1%|          | 1/100 [00:01<02:13,  1.35s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Se